## 1) Imports


In [5]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
from collections import defaultdict

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import h5py

# ── Graph ─────────────────────────────────────────────────────────────────────
import networkx as nx
from tqdm.notebook import tqdm


# ── Statistics ────────────────────────────────────────────────────────────────
from scipy.stats import mannwhitneyu
# ARI (Adjusted Rand Index): measures overlap between two partitions corrected for chance.
# NMI (Normalized Mutual Information): information-theoretic overlap, insensitive to community count.
# We compute both because ARI penalises splits/merges more harshly than NMI.
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ── Brain atlas & visualisation ───────────────────────────────────────────────
import nibabel as nib
from nilearn import datasets, plotting, image
from nilearn.maskers import NiftiLabelsMasker

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pathlib import Path
import os


warnings.filterwarnings('ignore')   # suppress nilearn cache / deprecation noise


---
## 2) Config


In [6]:
# walk upward from current dir till we find project folder
def find_project_root(marker='README.md'):
    path = Path(os.getcwd())
    for src in [path, *path.parents]:
        if (src / marker).exists():
            return src
    raise FileNotFoundError(
        f'Could not find the project root. Looked for {marker} starting from {path}.'
    )
    
PROJ_ROOT = find_project_root()
DB_PATH = PROJ_ROOT / 'SMA_data_processing' / 'cobre_combined_connectomes_database.h5'


GAMMA = 1.0 #louvain resolution, 1.0 is standard
THRESHOLD = 1e-3 # modularity convergence threshold
MAX_LEVELS=1000 #louvain lvls
RANDOM_SEED= 42 #for reproductibility

PALETTE = {'HC': '#4C9BE8', 'SCZ': '#E8734C'}


---
## 3) Louvain functions from Louvain.ipynb

- initial_assignment_communities: partition, every node is its own community
- delta_q: computes modularity gain for moving one node into a candidate community
- louvain_step_one: local optimisation phase (iter until no node moves)
- louvain_step_two: graph coarsening (collapse communities into super-nodes)
- modularity: computes standard Newman-Girvan Q score
- louvain: runs the full two-phase loop until convergence


In [ ]:
def initial_assignment_communities(G):
    return {node: node for node in G.nodes()}

def delta_q(G, node, target_community, communities, gamma=1.0, weight='weight'):
    degrees= dict(G.degree(weight=weight))
    two_m= sum(degrees.values())
    k_i_in= sum( #sum the weights of the neighbors of a node
        edge_data.get(weight,0)
        for neighbor, edge_data in G[node].items() # [node] is the lookup key, not a syntactic marker meaning "give me nodes" --> specific node
        if neighbor != node and communities[neighbor] ==target_community
    )
    
    k_i= degrees[node]
    
    sigma_tot=sum( #sum of degrees of all neighbors from the target_community
        degrees[n]
        for n in G.nodes() # all nodes
        if n!= node and communities[n] == target_community
    )
    
    return (k_i_in/two_m) - gamma * (sigma_tot * k_i) / (two_m **2)


def louvain_step_one(G, gamma=1.0, weight='weight'):
    """
    Phase 1 of Louvain — local modularity optimisation.
    """
    communities = initial_assignment_communities(G)

    # Pre-compute degrees once — O(edges)
    degrees = dict(G.degree(weight=weight))
    two_m   = sum(degrees.values())

    if two_m == 0:
        return communities   # empty graph — nothing to do

    # community_total_degree[c] = sum of degrees of all nodes in community c
    # Initialised with each node in its own community → equals its own degree
    community_total_degree = {node: degrees[node] for node in G.nodes()}

    moved = True
    while moved:
        moved = False

        for node in G.nodes():
            current_community = communities[node]
            k_i               = degrees[node]

            # --- temporarily remove node from its current community ----------
            communities[node]                          = -1
            community_total_degree[current_community] -= k_i

            # --- find unique neighboring communities -------------------------
            # We only need to consider communities of actual neighbors —
            # no need to scan all n nodes.
            neighbor_communities = {}   # {community_id: k_i_in (edge weight sum)}
            for neighbor, edge_data in G[node].items():
                nc = communities[neighbor]
                if nc == -1:
                    continue   # skip if neighbor is also isolated (shouldn't happen)
                w = edge_data.get(weight, 0)
                if nc not in neighbor_communities:
                    neighbor_communities[nc] = 0.0
                neighbor_communities[nc] += w

            # --- evaluate delta_q for each neighboring community -------------
            # Formula: delta_q = k_i_in/2m - gamma * sigma_tot * k_i / (2m)²
            # All quantities are now O(1) lookups — no inner loops.
            best_community = current_community
            best_dq        = 0.0   # only move if gain is strictly positive

            for tc, k_i_in in neighbor_communities.items():
                sigma_tot = community_total_degree.get(tc, 0.0)
                dq = (k_i_in / two_m) - gamma * (sigma_tot * k_i) / (two_m ** 2)
                if dq > best_dq:
                    best_dq        = dq
                    best_community = tc

            # --- place node into best community and update cache -------------
            communities[node]                         = best_community
            community_total_degree[best_community]    = \
                community_total_degree.get(best_community, 0.0) + k_i

            if best_community != current_community:
                moved = True

    return communities
    
    
def louvain_step_two(G, communities, weight= 'weight'): #each community becomes a super-node
    new_G= nx.graph()
    for uc in set(communities.values()):
        new_G.add_node(uc)
    
    for n1,n2,edge_data in G.edges(data=True):
        c1, c2 = communities[n1], communities[n2]
        w= edge_data.get(weight,0)
        if new_G.has_edge(c1,c2):
            new_G[c1][c2][weight] += w #adds current edge's weight to the running total
        else:
            new_G.add_edge(c1,c2, **{weight:w}) #1st time we encounter an edge between these 2 communities we create a super-edge between them with current weight as starting value
    return new_G


def modularity(G, communities, weight='weight', gamma=1.0):
    """
    Compute Newman-Girvan modularity Q in O(edges) instead of O(n²).

    Standard reformulation:
        Q = (1/2m) Σ_c [ L_c  -  gamma * D_c² / (2m) ]

    where for each community c:
        L_c = sum of weights of edges with BOTH endpoints in c
        D_c = sum of degrees (weighted) of all nodes in c

    This avoids the double loop over all node pairs entirely.
    """
    degrees = dict(G.degree(weight=weight))
    two_m   = sum(degrees.values())

    if two_m == 0:
        return 0.0

    # Accumulate L_c and D_c per community
    community_L = defaultdict(float)   # internal edge weight sum
    community_D = defaultdict(float)   # total degree sum

    for node, comm in communities.items():
        community_D[comm] += degrees[node]

    for u, v, data in G.edges(data=True):
        if communities[u] == communities[v]:
            w = data.get(weight, 0)
            community_L[communities[u]] += w   # each edge counted once

    Q = 0.0
    for comm in community_D:
        L_c = community_L.get(comm, 0.0)
        D_c = community_D[comm]
        # multiply L_c by 2 because each internal edge contributes to
        # both endpoints in the original double-sum formulation
        Q += (2 * L_c / two_m) - gamma * (D_c / two_m) ** 2

    return Q


        
def louvain(G, weight='weight', threshold=1e-3, max_levels=1000, gamma=1.0):
    """
    Full Louvain algorithm using the optimised phase 1 and O(edges) modularity.
    Returns dict {original_node: community_id} after convergence.

    Complexity: O(edges × n_passes × n_levels) — same class as NetworkX.
    Still pure Python so expect ~5-10x slower than NetworkX in wall time,
    but now finishes in minutes rather than days at n=1019.
    """
    current_G  = G
    levels     = 0
    prev_Q     = -1.0

    # Track mapping from original node → current super-node
    communities_original = {node: node for node in G.nodes()}

    while levels < max_levels:
        communities = louvain_step_one(current_G, gamma=gamma, weight=weight)
        curr_Q      = modularity(current_G, communities,
                                      weight=weight, gamma=gamma)

        if prev_Q != -1.0 and abs(curr_Q - prev_Q) < threshold: #if its insignificant, ignore
            break

        prev_Q    = curr_Q
        new_G     = louvain_step_two(current_G, communities, weight=weight)

        # Propagate mapping back to original nodes
        communities_original = {
            orig: communities[old_comm]
            for orig, old_comm in communities_original.items() #old_comm is actually the current community orig node is with after level 1
        }

        current_G = new_G
        levels   += 1

    return communities_original



## ------- ----------------------------- HELPER FUNCTIONS ---------------------------------
# ── Helper: dict → list-of-sets (format expected by nx.modularity) 
def communities_to_sets(communities_dict):
    grouped = defaultdict(set)
    for node, cid in communities_dict.items():
        grouped[cid].add(node)
    return list(grouped.values())


# ── Helper: dict → flat label array (format expected by sklearn metrics)
def communities_to_labels(communities_dict):
    """
    Returns a numpy array where index = node (assumes nodes are 0..N-1)
    and value = community id remapped to 0..K-1.
    """
    nodes = sorted(communities_dict.keys())
    raw   = np.array([communities_dict[n] for n in nodes])
    # remap community ids to contiguous integers
    unique = {v: i for i, v in enumerate(sorted(set(raw)))}
    return np.array([unique[v] for v in raw])


print('Louvain functions loaded.')

Louvain functions loaded.
